# Sign AR — Milestone 2: ASL Transformer Training

**Before running:** Set runtime to GPU (Runtime → Change runtime type → T4 or A100)

**Setup steps:**
1. Upload the entire `preprocessing/` folder from your Mac to Google Drive under `MyDrive/Sign_AR/`
2. Mount Drive in Cell 1
3. Run all cells in order

In [1]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/Sign_AR'

Mounted at /content/drive


In [2]:
# Cell 2: Check GPU
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : Tesla T4


In [3]:
# Cell 3: Copy model files and UNZIP dataset
import os, shutil, sys

MODEL_SRC  = os.path.join(DRIVE_ROOT, 'model') # changed from preprocessing/model
MODEL_DST  = '/content/model'
ZIP_PATH   = '/content/dataset.zip'  # <--- Points to direct upload
OUTPUT_DIR = '/content/output'

os.makedirs(MODEL_DST, exist_ok=True)
os.makedirs(os.path.join(MODEL_DST, 'checkpoints'), exist_ok=True)

# Copy python files from Drive
for fname in ['dataset.py', 'model.py', 'train.py', 'evaluate.py']:
    shutil.copy(os.path.join(MODEL_SRC, fname), os.path.join(MODEL_DST, fname))

# Unzip your direct upload
print('\nUnzipping dataset directly in Colab...')
!unzip -q "{ZIP_PATH}" -d /content/
print('Dataset unzipped!')

sys.path.insert(0, MODEL_DST)


Unzipping dataset directly in Colab...
Dataset unzipped!


In [4]:
# Cell 4: Sanity check — model shape
from model import ASLTransformer
import torch

model = ASLTransformer()
out   = model(torch.randn(4, 60, 255))
assert out.shape == (4, 100)
print(f'Forward pass OK: {out.shape}')
print(f'Parameters    : {model.count_parameters():,}')

/content/model/model.py:149: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Forward pass OK: torch.Size([4, 100])
Parameters    : 3,343,204


In [5]:
# Cell 5: Patch paths so dataset.py reads from Drive
import dataset as ds
ds.OUTPUT_DIR     = OUTPUT_DIR
ds.CLASS_MAP_PATH = os.path.join(MODEL_DST, 'class_map.json')
print(f'OUTPUT_DIR: {ds.OUTPUT_DIR}')

OUTPUT_DIR: /content/output


In [8]:
# Cell 6: TRAIN
# Running via CLI ensures it always reads the latest train.py if you edit it in Colab
!python /content/model/train.py


  ASL Transformer — Milestone 2 Training
  Device : cuda

[Dataset] Train: 2838 | Val: 359 | Test: 346
/content/model/model.py:149: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
[Model] Parameters: 3,343,204

[Sanity] Forward pass OK — output shape: torch.Size([2, 100])

/content/model/train.py:100: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
 Epoch      Loss   Val Acc          LR     Time  Note
--------------------------------------------------------------
    1/150    4.2321    12.53%    3.00e-04    0m03s  ⬆ best
    2/150    3.3890    33.70%    3.00e-04    0m04s  ⬆ best
    3/150    2.7594    46.52%    3.00e-04    0m03s  ⬆ best
    4/150    2.3704    49.58%    2.99e-04    0m03s  ⬆ best
    5/15

In [17]:
# Cell 7: Save checkpoint to Drive
CHECK_DST = os.path.join(MODEL_SRC, 'checkpoints', 'best_model.pt')
os.makedirs(os.path.dirname(CHECK_DST), exist_ok=True)
shutil.copy('/content/model/checkpoints/best_model.pt', CHECK_DST)
print(f'Checkpoint saved: {CHECK_DST}')

Checkpoint saved: /content/drive/MyDrive/Sign_AR/model/checkpoints/best_model.pt


In [9]:
# Cell 8: EVALUATE on test set
!python /content/model/evaluate.py


  ASL Transformer — Test Set Evaluation
  Device     : cuda
  Checkpoint : /content/model/checkpoints/best_model.pt

/content/model/model.py:149: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
[Model] Loaded from epoch 113 (val acc at save: 88.30%)

[Dataset] Train: 2838 | Val: 359 | Test: 346
  Top-1 Accuracy : 87.57%
  Top-5 Accuracy : 95.66%
  Test samples   : 346

  Gloss                         Correct   Total      Acc
  --------------------------------------------------------
  WHATFOR1                            1       4    25.0%  ██
  ANYONE                              1       3    33.3%  ███
  AXE1                                1       3    33.3%  ███
  BREAKFAST1                          2       4    50.0%  █████
  CANCEL1                             2       4    50.0%  █████
  FOREIGNER1                          2     

In [13]:
# Cell 9: Save confusion matrix to Drive
CM = '/content/model/confusion_matrix.png'
if os.path.exists(CM):
    shutil.copy(CM, os.path.join(MODEL_SRC, 'confusion_matrix.png'))
    print('Confusion matrix saved to Drive')

Confusion matrix saved to Drive
